In [1]:
from pathlib import Path
import pandas as pd

# Measured features
features = pd.read_csv("../../measurements/red_channel_df.csv")
print(features.shape)

# Read all annotation files
annotation_folder = Path("../../classification/annotations")

annotation_files = sorted(annotation_folder.glob("*.csv"))

annotations = pd.concat(
    (pd.read_csv(f) for f in annotation_files),
    ignore_index=True,
)

print(f"Loaded {len(annotation_files)} annotation files.")
print(annotations.shape)

# Merge on image and label
df = features.merge(
    annotations[["label", "image", "annotation"]],
    on=["label", "image"],
    how="inner",
)

print(df.shape)
print(df["annotation"].value_counts())

(42836, 113)
Loaded 2 annotation files.
(88, 7)
(88, 114)
annotation
0    45
1    43
Name: count, dtype: int64


In [2]:
annotations

,image,point_index,annotation,label,z,y,x
0,1.3_P301S+PLA_BS_AT8_E2_first bottom_2026-04-2...,0,1,43,2.0,666.765567,777.741889
1,1.3_P301S+PLA_BS_AT8_E2_first bottom_2026-04-2...,1,1,490,2.0,721.580419,583.453689
2,1.3_P301S+PLA_BS_AT8_E2_first bottom_2026-04-2...,2,1,649,2.0,651.250243,739.272165
3,1.3_P301S+PLA_BS_AT8_E2_first bottom_2026-04-2...,3,1,494,3.0,741.798422,499.338005
4,1.3_P301S+PLA_BS_AT8_E2_first bottom_2026-04-2...,4,1,659,3.0,700.462243,533.032202
...,...,...,...,...,...,...,...
83,1.3_P301S+PLA_MC_AT8_E2_first bottom_2026-04-2...,21,0,236,4.0,1793.074656,1854.944571
84,1.3_P301S+PLA_MC_AT8_E2_first bottom_2026-04-2...,22,0,1102,2.0,1743.165035,1871.695060
85,1.3_P301S+PLA_MC_AT8_E2_first bottom_2026-04-2...,23,0,1088,2.0,1716.842838,1837.168542
86,1.3_P301S+PLA_MC_AT8_E2_first bottom_2026-04-2...,24,0,1009,2.0,1292.796509,1989.222395


In [3]:
df_train = df.dropna(subset=["annotation"])

from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df_train,
    test_size=0.2,      # 20% test, 80% train
    random_state=42,    # reproducible split
    stratify=df_train["annotation"],  # preserve class balance
)

In [6]:
drop_columns = [
    "annotation",
    "label",
    "image_name",
    "valid",
]

feature_cols = [c for c in df_train.columns if c not in drop_columns]

X_train = df_train[feature_cols].select_dtypes(include="number")
X_train.fillna(X_train.median())
y_train = df_train["annotation"]

X_test = df_test[feature_cols].select_dtypes(include="number")
X_test.fillna(X_test.median())
y_test = df_test["annotation"]

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [8]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_,
})

importance = importance.sort_values(
    "importance",
    ascending=False,
)

print(importance.head(20))

                              feature  importance
66                 red_blur_2p0um_p80    0.040000
72                 red_dilated_r2_p95    0.038111
90  red_puncta_structure_p99_fraction    0.038000
80          red_dilated_r2_inside_p95    0.036000
85            red_puncta_p99_fraction    0.036000
16                 red_blur_0p5um_std    0.034112
31                 red_blur_0p5um_p95    0.034000
48                 red_blur_1p0um_p90    0.034000
32                 red_blur_0p5um_p98    0.034000
13                  intensity_p98_red    0.030410
29                 red_blur_0p5um_p90    0.030000
64                 red_blur_2p0um_p60    0.028111
12                  intensity_p95_red    0.028000
49                 red_blur_1p0um_p92    0.026000
11                  intensity_p92_red    0.026000
51                 red_blur_1p0um_p98    0.024111
14                red_blur_0p5um_mean    0.024000
50                 red_blur_1p0um_p95    0.024000
47                 red_blur_1p0um_p80    0.024000


In [10]:
all_features = features.copy()

X_all = (
    all_features
    .drop(columns=[c for c in drop_columns if c in all_features.columns])
    .select_dtypes(include="number")
    .fillna(X_train.median())
)

all_features["prediction"] = model.predict(X_all)
all_features["probability"] = model.predict_proba(X_all)[:,1]

all_features.to_csv(
    "../../classification/output/props_with_predictions.csv",
    index=False,
)

In [ ]:
import numpy as np
import pandas as pd
import tifffile
import napari
import skimage.transform
from bioio import BioImage


# image_name = "1.3_P301S+PLA_BS_AT8_E2_first bottom_2026-04-22_14.32.11_FusionStitcher"
image_name = "64U-J9910_CHST11KO_BS_AT8_delta4_first_2026-04-23_12.32.20_FusionStitcher"
region = "brainstem"
MASK_PATH = f"L:/Algernon/530/Sarah/segmentation_output_pseudo3d_260611/{region}/{image_name}_mask_3d.tif"
IMAGE_PATH = f"N:/01_scientific_data/Sarah_Kralova_brain_slice_data/cellpose + far-red channel/{region}/{image_name}.ims"

CSV_PATH = "../../classification/output/props_with_predictions.csv"

BIOIO_SCENE_INDEX = 0

CH_DAPI = 0
CH_GREEN = 1
CH_RED = 2
CH_FARRED = 3

CLASSIFICATION_COLUMN = "prediction"
# CLASSIFICATION_COLUMN = "annotation"


def load_image_czyx(path, scene_index=0):
    bio = BioImage(str(path))
    bio.set_scene(bio.scenes[scene_index])
    return bio.get_image_data("CZYX", T=0)


def downscale_image_to_mask_shape(image_czyx, mask_shape):
    if image_czyx.shape[1:] == mask_shape:
        return image_czyx

    target_shape = (image_czyx.shape[0],) + tuple(mask_shape)

    print(f"Downscaling image from {image_czyx.shape} to {target_shape}")

    return skimage.transform.resize(
        image_czyx,
        target_shape,
        order=1,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(image_czyx.dtype, copy=False)


def make_classification_mask(mask, table, class_column):
    table = table.dropna(subset=[class_column]).copy()

    table["label"] = table["label"].astype(np.uint32)
    table[class_column] = table[class_column].astype(int)

    max_label = int(mask.max())
    lut = np.zeros(max_label + 1, dtype=np.uint8)

    for label_id, cls in zip(
        table["label"].to_numpy(),
        table[class_column].to_numpy(),
    ):
        if 0 < label_id <= max_label:
            # background = 0
            # class 0 = 1
            # class 1 = 2
            lut[int(label_id)] = int(cls) + 1

    return lut[mask]


def add_channel(viewer, image, name, colormap):
    viewer.add_image(
        image,
        name=name,
        colormap=colormap,
        blending="additive",
        contrast_limits=np.percentile(image, [1, 99.5]),
    )


def main():
    image_czyx = load_image_czyx(IMAGE_PATH, BIOIO_SCENE_INDEX)

    mask = tifffile.imread(MASK_PATH).squeeze().astype(np.uint32)

    image_czyx = downscale_image_to_mask_shape(
        image_czyx,
        mask.shape,
    )

    table = pd.read_csv(CSV_PATH)

    if CLASSIFICATION_COLUMN not in table.columns:
        raise ValueError(
            f"Column '{CLASSIFICATION_COLUMN}' not found in table. "
            f"Available columns are: {list(table.columns)}"
        )

    classification_mask = make_classification_mask(
        mask,
        table,
        CLASSIFICATION_COLUMN,
    )

    viewer = napari.Viewer()

    add_channel(viewer, image_czyx[CH_DAPI], "DAPI", "blue")
    add_channel(viewer, image_czyx[CH_GREEN], "green", "green")
    add_channel(viewer, image_czyx[CH_RED], "red", "red")
    add_channel(viewer, image_czyx[CH_FARRED], "far-red", "magenta")

    viewer.add_labels(
        classification_mask,
        name=f"classification: {CLASSIFICATION_COLUMN}",
        opacity=0.55,
    )

    viewer.add_labels(
        np.max(classification_mask, axis=0).astype(np.uint32),
        name=f"classification: {CLASSIFICATION_COLUMN} (max projection)",
        opacity=0.55,
    )

    viewer.add_image(
        np.max(image_czyx[CH_RED], axis=0),
        name=f"MIP red",
        colormap="red",
        blending="additive",
        contrast_limits=np.percentile(image_czyx[CH_RED], [1, 99.5]),
    )

    viewer.add_labels(
        mask,
        name="original cell labels",
        opacity=0.2,
        visible=False,
    )

    napari.run()


if __name__ == "__main__":
    main()